# Ensamble Final — Pesos Óptimos por Scipy

Este notebook carga las predicciones de validación de CADA modelo y encuentra los
pesos que maximizan el Macro ROC-AUC en ese conjunto de validación compartido
(random_state=42, test_size=0.20).

**Modelos disponibles para combinar:**
1. TF-IDF avanzado (word + char n-grams, multi-C)  → `pred_tfidf_avanzado.csv`
2. DeBERTa-v3-base                                  → `pred_deberta.csv` / `pred_deberta_full.csv`
3. Primer modelo del compañero                       → `submission_primer_modelo.csv` (si disponible)

Los pesos se optimizan usando **Nelder-Mead** (optimización sin gradientes) sobre
el espacio simplex (pesos ≥ 0, suma = 1).


In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import ast


## 1. Reconstruir el split de validación (mismo seed que cada modelo)

In [ ]:
# Cargar datos originales solo para reconstruir y_val
train_raw = pd.read_csv(
    'https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip',
    encoding='UTF-8', index_col=0
)
test_raw = pd.read_csv(
    'https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip',
    encoding='UTF-8', index_col=0
)

train_raw['genres'] = train_raw['genres'].apply(ast.literal_eval)
mlb = MultiLabelBinarizer()
y   = mlb.fit_transform(train_raw['genres'])

idx_all = np.arange(len(train_raw))
_, idx_val = train_test_split(idx_all, test_size=0.20, random_state=42)
y_val = y[idx_val]

print(f"Tamaño validación: {len(y_val)}")
print(f"Clases: {list(mlb.classes_)}")


## 2. Cargar predicciones de validación

In [ ]:
# ── Cargar cada archivo de predicciones de validación ─────────────────────
# Cada modelo debe haber guardado un .npy con sus predicciones en el split de val.
# Si no existe, usamos las predicciones de test como proxy (menos preciso).

model_names = []
val_preds   = []
test_preds  = []

COLS = ['p_Action','p_Adventure','p_Animation','p_Biography','p_Comedy',
        'p_Crime','p_Documentary','p_Drama','p_Family','p_Fantasy',
        'p_Film-Noir','p_History','p_Horror','p_Music','p_Musical',
        'p_Mystery','p_News','p_Romance','p_Sci-Fi','p_Short',
        'p_Sport','p_Thriller','p_War','p_Western']

import os

def load_model(name, val_npy, test_csv):
    if os.path.exists(val_npy) and os.path.exists(test_csv):
        vp = np.load(val_npy)
        tp = pd.read_csv(test_csv, index_col='ID')[COLS].values
        model_names.append(name)
        val_preds.append(vp)
        test_preds.append(tp)
        auc = roc_auc_score(y_val, vp, average='macro')
        print(f"{name:30s}  val ROC-AUC = {auc:.4f}  shape={vp.shape}")
    else:
        print(f"[SKIP] {name} — archivos no encontrados: {val_npy}, {test_csv}")

load_model('TF-IDF Avanzado',  'val_preds_tfidf_avanzado.npy', 'pred_tfidf_avanzado.csv')
load_model('DeBERTa-v3-base',  'val_preds_deberta.npy',        'pred_deberta.csv')

print(f"\nModelos cargados: {len(model_names)}")


## 3. Optimización de pesos — Nelder-Mead en el simplex

El espacio de búsqueda es el **simplex** estándar: todos los pesos ≥ 0 y su suma = 1.
Nelder-Mead es robusto frente a discontinuidades y funciona bien con pocas variables
(1 peso por modelo).


In [ ]:
n = len(val_preds)
val_arr  = np.array(val_preds)   # (n_models, n_val, n_classes)
test_arr = np.array(test_preds)  # (n_models, n_test, n_classes)

def neg_auc(weights):
    w = np.array(weights)
    w = np.clip(w, 0, None)
    w = w / w.sum()
    combined = (val_arr * w[:, None, None]).sum(axis=0)
    return -roc_auc_score(y_val, combined, average='macro')

# Punto de partida: pesos iguales
w0 = np.ones(n) / n
print(f"AUC pesos iguales: {-neg_auc(w0):.4f}")

result = minimize(
    neg_auc,
    x0=w0,
    method='Nelder-Mead',
    options={'maxiter': 5000, 'xatol': 1e-6, 'fatol': 1e-6}
)

w_opt = np.clip(result.x, 0, None)
w_opt = w_opt / w_opt.sum()

print("\n=== Pesos óptimos ===")
for name, w in zip(model_names, w_opt):
    print(f"  {name:30s}: {w:.4f} ({w*100:.1f}%)")

auc_opt = -result.fun
print(f"\nAUC ensamble óptimo (val): {auc_opt:.4f}")


## 4. Submission final

In [ ]:
# Aplicar pesos óptimos a predicciones de test
y_ensemble = (test_arr * w_opt[:, None, None]).sum(axis=0)

sub = pd.DataFrame(y_ensemble, index=test_raw.index, columns=COLS)
sub.to_csv('pred_ensemble_final.csv', index_label='ID')
print("Submission guardado: pred_ensemble_final.csv")
sub.head()


## 5. Análisis por género (diagnóstico)

In [ ]:
print("=== AUC por género ===")
auc_por_clase = []
for i, g in enumerate(mlb.classes_):
    if len(np.unique(y_val[:, i])) > 1:
        combined_val = (val_arr * w_opt[:, None, None]).sum(axis=0)
        a = roc_auc_score(y_val[:, i], combined_val[:, i])
    else:
        a = float('nan')
    auc_por_clase.append(a)
    print(f"  {g:15s}: {a:.4f}  (positivos en val: {y_val[:, i].sum()})")

print(f"\nMacro AUC: {np.nanmean(auc_por_clase):.4f}")
